In [1]:
import numpy as np
import pandas as pd

In [2]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder

In [3]:
df = pd.read_csv('D:\\ML-Arena\\Data_Preprocessing\\covid_toy.csv')

In [4]:
df.head()

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No


In [5]:
df.isnull().sum()

age           0
gender        0
fever        10
cough         0
city          0
has_covid     0
dtype: int64

In [6]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(df.drop(columns='has_covid'), df['has_covid'], test_size=0.2)

In [7]:
X_train

,age,gender,fever,cough,city
51,11,Female,100.0,Strong,Kolkata
67,65,Male,99.0,Mild,Bangalore
57,49,Female,99.0,Strong,Bangalore
34,74,Male,102.0,Mild,Mumbai
45,72,Male,99.0,Mild,Bangalore
...,...,...,...,...,...
18,64,Female,98.0,Mild,Bangalore
25,23,Male,NaN,Mild,Mumbai
77,8,Female,101.0,Mild,Kolkata
9,64,Female,101.0,Mild,Delhi


## Without using Column Transformer

In [8]:
# Adding simple imputer to fever column
si = SimpleImputer()
X_train_fever = si.fit_transform(X_train[['fever']])

# Also test the data
X_test_fever = si.transform(X_test[['fever']])

X_train_fever.shape

(80, 1)

In [9]:
# Ordinalencoding -> cough
oe = OrdinalEncoder(categories=[['Mild', 'Strong']])
X_train_cough = oe.fit_transform(X_train[['cough']])

# Also test the data
X_test_cough = oe.transform(X_test[['cough']])

X_train_cough.shape

(80, 1)

In [11]:
# OneHotEncoding -> gender, city
ohe = OneHotEncoder(drop = 'first', sparse_output=False)
X_train_gender_city = ohe.fit_transform(X_train[['gender', 'city']])

# Also test the data
X_test_gender_city = ohe.transform(X_test[['gender', 'city']])
X_train_gender_city.shape

(80, 4)

In [13]:
# Extracting Age
X_train_age = X_train.drop(columns=['gender', 'fever', 'cough', 'city']).values

# Also test the data
X_test_age = X_test.drop(columns=['gender', 'fever', 'cough', 'city']).values

X_train_age.shape

(80, 1)

In [14]:
X_train_transformed = np.concatenate([X_train_age, X_train_fever, X_train_cough, X_train_gender_city], axis=1)

# Also test the data
X_test_transformed = np.concatenate([X_test_age, X_test_fever, X_test_gender_city, X_test_cough], axis=1)

X_train_transformed.shape

(80, 7)

## Using Column Transformer

In [15]:
from sklearn.compose import ColumnTransformer

In [16]:
transformer = ColumnTransformer(transformers=[
    ('tnf1',SimpleImputer(),['fever']),
    ('tnf2',OrdinalEncoder(categories=[['Mild','Strong']]),['cough']),
    ('tnf3',OneHotEncoder(sparse_output=False,drop='first'),['gender','city'])
],remainder='passthrough')

In [17]:
transformer.fit_transform(X_train).shape

(80, 7)

In [18]:
transformer.fit_transform(X_test).shape

(20, 7)